In [ ]:
# !pip install azure-ai-documentintelligence azure-core python-dotenv langchain langchain-core langchain-community --upgrade

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

True

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

endpoint = os.environ['DOCUMENT_INTELLIGENCE_ENDPOINT']
key = os.environ['DOCUMENT_INTELLIGENCE_KEY']

def analyze_resume():
    client = DocumentIntelligenceClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(key)
    )

    file_path = "forms/form3.png"  # 👈 your PDF file name

    with open(file_path, "rb") as f:
        poller = client.begin_analyze_document(
            "prebuilt-read",  # best for resumes
            body=f
        )

    result = poller.result()
    return result.content

form_text = analyze_resume()
form_text

'INSURANCE CLAIM FORM\nClaim ID: CLM1002\nDate: 16/01/2025\n1. POLICY HOLDER DETAILS\nName:\nPriya Mehta\nPolicy Number:\nPOL789012\nCustomer ID:\nC002\nPhone Number:\n9123456780\nEmail:\npriya.mehta@email.com\nAddress:\n45, Green Avenue, Mumbai - 400001\n2. CLAIM DETAILS\nClaim Type:\nAccident\nLocation:\nMumbai\nClaim Amount:\n75000\nDescription of Incident:\nTwo wheeler accident causing leg fracture.\nDate of Incident:\n14/01/2025\n3. PREVIOUS CLAIMS\nNumber of Previous Claims:\n2\n4. DECLARATION\nI hereby declare that the information provided above is true and correct to the best of my knowledge.\nSignature: Priya Mehta'

In [120]:
import ast
model = init_chat_model(
    model='mistral-medium'
)

prompt = f"""
Extract the following fields from the resume text.

Return ONLY valid JSON in this format:
{{
  "Customer_ID": <str>,
  "Claim_Amount": <int>,
  "Claim_Type": <str>,
  "Location": <str>,
  "Previous_Claims": <int>
}}

Rules:
- Extract only relevant skills
- No extra text, only JSON

Resume:
{form_text}
"""

res = model.invoke(prompt).content
from rich.console import Console
from rich.markdown import Markdown
console = Console()

res = res.replace("```json","").replace("```","").replace('\n',"").replace("  ","")
res = ast.literal_eval(res)
print(res)

{'Customer_ID': 'C002', 'Claim_Amount': 75000, 'Claim_Type': 'Accident', 'Location': 'Mumbai', 'Previous_Claims': 2}


In [118]:
import re
def print_result(response):

# ── 1. String fields ──────────────────────────────────────────
    customer_id  = re.search(r'"Customer_ID"\s*:\s*"([^"]+)"',  response).group(1)
    claim_type   = re.search(r'"Claim_Type"\s*:\s*"([^"]+)"',   response).group(1)
    location     = re.search(r'"Location"\s*:\s*"([^"]+)"',     response).group(1)

# ── 2. Numeric fields ─────────────────────────────────────────
    claim_amount     = int(re.search(r'"Claim_Amount"\s*:\s*(\d+)',          response).group(1))
    previous_claims  = int(re.search(r'"Previous_Claims"\s*:\s*(\d+)',       response).group(1))
    scored_prob      = float(re.search(r'"Scored Proailities"\s*:\s*([\d.]+)', response).group(1))

# ── 3. Boolean field ──────────────────────────────────────────
    scored_labels_raw = re.search(r'"Scored Laels"\s*:\s*(true|false)', response).group(1)
    scored_labels     = scored_labels_raw == "true"

# ── 4. Print all extracted fields ────────────────────────────
    print("Customer_ID        :", customer_id)
    print("Claim_Amount       :", claim_amount)
    print("Claim_Type         :", claim_type)
    print("Location           :", location)
    print("Previous_Claims    :", previous_claims)
    print("Scored Labels      :", scored_labels)
    print("Scored Probability :", scored_prob.__round__(4))

In [119]:
import urllib.request
import json

url = os.environ.get("ML_ENDPOINT")
api_key = os.environ.get("ML_API_KEY")

if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

data = {"Inputs": {"input1": [res]}, "GlobalParameters": {}}

body = str.encode(json.dumps(data))
headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)
    result = str(response.read())
    result = result.replace("b","")
    print_result(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

Customer_ID        : C002
Claim_Amount       : 75000
Claim_Type         : Accident
Location           : Mumai
Previous_Claims    : 2
Scored Labels      : True
Scored Probability : 0.9037
